In [ ]:
import sys; sys.path.append('..')
import MeshFEM, mesh, mesh_energy, param_utils, viewer, benchmark
import numpy as np
import sim_utils

import matplotlib
from matplotlib import pyplot as plt
import visualization

import newton_flow

In [ ]:
# m = param_utils.load('../../models/lucy.msh.xz')
# m = param_utils.load('../../models/cow2Disc.msh')
m = param_utils.load('../../models/hilbert_curve.msh.xz')
tutte_uv = param_utils.tutteInitialization(m)
v = mesh_energy.NodalVars(m, 2)
v.setVars(tutte_uv.ravel())
m_2d = mesh.Mesh(np.zeros_like(tutte_uv), m.elements())
m_2d.reembedElements(m.vertices())
nf = newton_flow.symmetric_dirichlet(m_2d, v)

In [ ]:
import fast_newton_flow

In [ ]:
fnf = fast_newton_flow.symmetric_dirichlet(m_2d, v)

In [ ]:
nf.projectionSmoothingEpsilon = 0 # 1e-4 # 1e-8

In [ ]:
nf.objectiveAtVars(v.getVars())

In [ ]:
fnf.objective()

In [ ]:
import py_newton_optimizer
prob = py_newton_optimizer.NewtonMultiobjectiveProblem(v, [nf])

In [ ]:
# Nullspace pinning strategy
FIX_VARS = False
if FIX_VARS:
    # fv = sim_utils.getBBoxVars(m_rest, sim_utils.BBoxFace.MIN_X)
    # prob.setFixedVars(fv)
    import elastic_solid, energy
    es = elastic_solid.ElasticSolid(m_rest, energy.CommonNeoHookeanYoungPoisson(2, 1, 0.3))
    es.setDeformedPositions(m_defo.vertices())
    pin_vars, _ = es.prepareRigidMotionPins()
    v.setVars(es.getVars())
    prob.setFixedVars(pin_vars)
else:
    # prob.hessianShift = 1e-5
    # nf.elementHessianShift = 1e-8
    # nf.elementHessianShift = 1e-8
    prob.hessianShift = 1e-10
    prob.useRelativeHessianShift = False

In [ ]:
import newton_flow_utils

In [ ]:
constant_speed = True
always_project = True

In [ ]:
opt = prob.optimizer()
opt.options.hessianProjectionController.startWithProjectionActive = False
opt.options.hessianProjectionController.numProjectionStepsBeforeDisable = 1
opt.options.hessianProjectionController.numConsecutiveIndefiniteStepsBeforeEnable = 0
if always_project: opt.options.hessianProjectionController = py_newton_optimizer.HessianProjectionAlways()

In [ ]:
opt.options.niter = 10
opt.optimize()
opt.update_factorizations()

In [ ]:
import parallelism
parallelism.set_max_num_tbb_threads(1)

In [ ]:
max_degree = 19

In [ ]:
# coeffs = nf.computeTaylorCoefficients(opt.hessian_factorization, projectHessian = True, degree=max_degree) # warm up
# benchmark.reset()
# coeffs = nf.computeTaylorCoefficients(opt.hessian_factorization, projectHessian = True, degree=max_degree)
# benchmark.report()
# aos_times = [benchmark.totalTime(f'order {i}$') for i in range(1, max_degree+1)]

In [ ]:
coeffs = nf.computeTaylorCoefficients(opt.hessian_factorization, projectHessian = False, degree=max_degree) # warm up
benchmark.reset()
coeffs = nf.computeTaylorCoefficients(opt.hessian_factorization, projectHessian = False, degree=max_degree)
benchmark.report()
aos_times = [benchmark.totalTime(f'order {i}$') for i in range(1, max_degree+1)]

In [ ]:
# coeffs = nf.computeTaylorCoefficientsArclen(opt.hessian_factorization, projectHessian = False, degree=max_degree) # warm up
# benchmark.reset()
# coeffs = nf.computeTaylorCoefficientsArclen(opt.hessian_factorization, projectHessian = False, degree=max_degree)
# benchmark.report()
# aos_times = [benchmark.totalTime(f'order {i}$') for i in range(1, max_degree+1)]

In [ ]:
fcoeffs = fnf.computeTaylorCoefficients(opt.hessian_factorization, projectHessian = False, degree=max_degree, arclen=True) # warm up
benchmark.reset()
# max_degree = 100
fcoeffs = fnf.computeTaylorCoefficients(opt.hessian_factorization, projectHessian = False, degree=max_degree, arclen=True)
benchmark.report()
soa_times = [benchmark.totalTime(f'P upgrade {i}$') for i in range(1, max_degree+1)]

In [ ]:
fcoeffs = fnf.computeTaylorCoefficients(opt.hessian_factorization, projectHessian = False, degree=max_degree, arclen=False) # warm up

In [ ]:
plt.plot(aos_times)
plt.plot(soa_times);

In [ ]:
def relerror(a, b):
    return np.linalg.norm(a - b) / np.linalg.norm(a)

In [ ]:
for i in range(max_degree):
    print(relerror(coeffs[i], fcoeffs[i]))